# 02 - LLM Fine-tuning (LoRA + FSDP)

Uses `TransformersTrainer(func=train_llm)` from `kubeflow.trainer.rhai` with the built-in
`torch-distributed` ClusterTrainingRuntime. Enables automatic progression tracking,
JIT checkpointing with S3 auto-resume, and metrics export via HTTP on port 28080.

| Step | What happens |
|------|-------------|
| 1 | SDK serializes `train_llm` → TrainJob CR with progression + checkpoint instrumentation |
| 2 | Controller creates JobSet (N pods, torchrun, PET_* envs) |
| 3 | LoRA + FSDP full_shard across nodes/GPUs with live metrics |
| 4 | SDK periodic checkpoints → S3 (auto-resume on restart) |
| 5 | Rank-0 saves final LoRA adapter to S3 + registers in MLflow |

In [ ]:
%pip install -q kubeflow --no-cache-dir \
    --index-url https://console.redhat.com/api/pypi/public-rhai/rhoai/3.3/cuda12.9-ubi9/simple/
%pip install -q kubernetes "fsspec[s3]" s3fs boto3

In [ ]:
import os

# Papermill parameters — overridden at runtime via -p flags (values come from .env)
NAMESPACE = os.environ.get("NAMESPACE", "smartshop")
RUNTIME = "torch-distributed"
DATA_DIR = os.environ.get("DATA_DIR", f"s3://{os.environ.get('S3_FEATURES_BUCKET', 'smartshop-features')}/llm_data")
OUTPUT_DIR = os.environ.get("OUTPUT_DIR", f"s3://{os.environ.get('S3_MODELS_BUCKET', 'smartshop-models')}/llm-adapter")
BASE_MODEL = os.environ.get("LLM_BASE_MODEL", "mistralai/Mistral-7B-Instruct-v0.3")
MAX_FILES = int(os.environ.get("LLM_MAX_FILES", "3"))
EPOCHS = int(os.environ.get("LLM_TRAIN_EPOCHS", "1"))
MAX_STEPS = int(os.environ.get("LLM_MAX_STEPS", "1500"))
BATCH_SIZE = int(os.environ.get("LLM_TRAIN_BATCH_SIZE", "4"))
GRADIENT_ACCUMULATION = int(os.environ.get("LLM_GRAD_ACCUM", "2"))
LR = 2e-4
LORA_R = 16
LORA_ALPHA = 32
MAX_SEQ_LENGTH = 2048
NUM_NODES = int(os.environ.get("LLM_TRAIN_NODES", "4"))
GPUS_PER_NODE = int(os.environ.get("LLM_GPUS_PER_NODE", "2"))
MINIO_ENDPOINT = os.environ.get("MINIO_ENDPOINT", os.environ.get("AWS_ENDPOINT_URL_S3", ""))
MLFLOW_TRACKING_URI = os.environ.get("MLFLOW_TRACKING_URI", "")
CLUSTER_DOMAIN = os.environ.get("OC_CLUSTER_DOMAIN", "")
TIMEOUT_SECONDS = 7200
S3_CREDENTIALS_SECRET = "smartshop-credentials"
MLFLOW_SECRET = "smartshop-mlflow-token"
HF_SECRET = "hf-credentials"

## Authentication

In [ ]:
import os
import warnings
import urllib3

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
warnings.filterwarnings("ignore", message=".*Unverified HTTPS.*")

IN_CLUSTER = os.path.exists("/var/run/secrets/kubernetes.io/serviceaccount/token")
K8S_TOKEN = os.getenv("K8S_TOKEN", "")
K8S_API = os.getenv("K8S_API_SERVER", "")

if not K8S_TOKEN and IN_CLUSTER:
    with open("/var/run/secrets/kubernetes.io/serviceaccount/token") as f:
        K8S_TOKEN = f.read().strip()
    K8S_API = "https://kubernetes.default.svc"

if not K8S_API and CLUSTER_DOMAIN:
    K8S_API = f"https://api.{CLUSTER_DOMAIN.replace('apps.', '')}:6443"

print(f"K8S API: {K8S_API or '(in-cluster)'}")
print(f"Token: {K8S_TOKEN[:20]}..." if K8S_TOKEN else "No token — using in-cluster defaults")
print(f"In-cluster: {IN_CLUSTER}")

## Initialize TrainerClient

In [ ]:
from kubernetes import client as k8s
from kubeflow.trainer import TrainerClient
from kubeflow.common.types import KubernetesBackendConfig

cfg = None
if K8S_TOKEN:
    cfg = k8s.Configuration()
    if K8S_API:
        cfg.host = K8S_API
    cfg.verify_ssl = False
    cfg.api_key = {'authorization': f'Bearer {K8S_TOKEN}'}

trainer = TrainerClient(
    KubernetesBackendConfig(namespace=NAMESPACE, client_configuration=cfg)
)

runtime = trainer.get_runtime(RUNTIME)
print(f'Runtime: {RUNTIME}')

## Define Training Function

Self-contained LoRA + FSDP fine-tuning function. The SDK serializes this into the pod.
`torchrun` + `PET_*` env vars are injected automatically by the controller.
FSDP `full_shard` distributes model parameters across all GPUs (ZeRO-3 equivalent).

In [ ]:
def train_llm(
    data_dir: str,
    output_dir: str,
    base_model: str,
    epochs: int = 1,
    max_steps: int = -1,
    batch_size: int = 4,
    gradient_accumulation: int = 4,
    lr: float = 2e-4,
    lora_r: int = 16,
    lora_alpha: int = 32,
    max_seq_length: int = 2048,
    max_files: int = 3,
):
    """LoRA fine-tuning with FSDP full_shard. Serialized by Kubeflow SDK into the pod."""
    import os
    import tempfile

    import fsspec
    import torch
    from datasets import load_dataset
    from peft import LoraConfig, get_peft_model
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from trl import SFTConfig, SFTTrainer

    rank = int(os.environ.get("RANK", 0))
    local_rank = int(os.environ.get("LOCAL_RANK", 0))
    world_size = int(os.environ.get("WORLD_SIZE", 1))

    if rank == 0:
        print(f"Fine-tuning {base_model} with LoRA + FSDP full_shard")
        print(f"PET_NNODES={os.environ.get('PET_NNODES')} | World size: {world_size}")
        print(f"Data: {data_dir} | Output: {output_dir}")

    # --- Load datasets from S3 ---
    s3_endpoint = os.environ.get("AWS_ENDPOINT_URL_S3", "")
    storage_opts = {}
    if data_dir.startswith("s3://") and s3_endpoint:
        storage_opts = {
            "key": os.environ.get("AWS_ACCESS_KEY_ID", ""),
            "secret": os.environ.get("AWS_SECRET_ACCESS_KEY", ""),
            "client_kwargs": {"endpoint_url": s3_endpoint},
        }

    def _load_split(split):
        split_path = data_dir.rstrip("/") + "/" + split
        fs, _ = fsspec.core.url_to_fs(split_path, **({"endpoint_url": s3_endpoint} if s3_endpoint else {}))
        if not fs.exists(split_path):
            return None
        entries = sorted(fs.ls(split_path, detail=False))
        part_files = [e for e in entries if
                      os.path.basename(e).startswith("part-") or
                      os.path.basename(e).endswith(".txt") or
                      os.path.basename(e).endswith(".jsonl")]
        if max_files and max_files > 0:
            part_files = part_files[:max_files]
        if data_dir.startswith("s3://") and part_files and not part_files[0].startswith("s3://"):
            part_files = [f"s3://{f}" for f in part_files]
        if rank == 0:
            print(f"  {split}: {len(part_files)} files")
        return load_dataset("json", data_files=part_files, split="train",
                            storage_options=storage_opts or None)

    train_dataset = _load_split("train")
    val_dataset = _load_split("val")
    if train_dataset is None:
        raise RuntimeError(f"No training data found at {data_dir}/train")
    if rank == 0:
        print(f"Train: {len(train_dataset):,} examples")
        if val_dataset:
            print(f"Val: {len(val_dataset):,} examples")

    def format_instruction(example):
        instruction = example.get("instruction", "")
        input_text = example.get("input", "")
        output_text = example.get("output", "")
        if input_text:
            prompt = f"[INST] {instruction}\n\n{input_text} [/INST]"
        else:
            prompt = f"[INST] {instruction} [/INST]"
        return f"{prompt} {output_text}" if output_text else prompt

    # --- Model download synchronization (local_rank 0 fetches, others wait) ---
    _lock = os.path.join(os.environ.get("HF_HOME", "/tmp/hf_home"), ".model_ready")
    if local_rank == 0:
        from huggingface_hub import snapshot_download
        snapshot_download(base_model)
        open(_lock, "w").close()
        if rank == 0:
            print("local_rank 0: model weights cached")
    else:
        import time as _tw
        while not os.path.exists(_lock):
            _tw.sleep(2)

    # Load in bf16 without device_map — FSDP handles sharding & placement
    model = AutoModelForCausalLM.from_pretrained(
        base_model,
        torch_dtype=torch.bfloat16,
        trust_remote_code=True,
    )

    peft_config = LoraConfig(
        r=lora_r,
        lora_alpha=lora_alpha,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
    )
    model = get_peft_model(model, peft_config)
    if rank == 0:
        model.print_trainable_parameters()

    tokenizer = AutoTokenizer.from_pretrained(base_model, trust_remote_code=True)
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    # --- MLflow setup ---
    import time as _time
    import logging
    import mlflow
    logging.getLogger("mlflow.tracing.export.mlflow_v3").setLevel(logging.ERROR)

    use_mlflow = False
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    trainable_pct = 100 * trainable_params / total_params

    if rank == 0:
        try:
            os.environ.setdefault("MLFLOW_TRACKING_INSECURE_TLS", "true")
            workspace = os.environ.pop("MLFLOW_WORKSPACE", None)
            tracking_uri = os.environ.get("MLFLOW_TRACKING_URI", "").rstrip("/")
            if tracking_uri.endswith("/mlflow"):
                tracking_uri = tracking_uri[:-len("/mlflow")]
                os.environ["MLFLOW_TRACKING_URI"] = tracking_uri
            if tracking_uri:
                mlflow.set_tracking_uri(tracking_uri)
            if workspace:
                from mlflow.utils import rest_utils as _ru
                _orig_http = _ru.http_request
                def _ws_http(*a, **kw):
                    h = kw.get("extra_headers", {}) or {}
                    h["X-MLflow-Workspace"] = workspace
                    kw["extra_headers"] = h
                    return _orig_http(*a, **kw)
                _ru.http_request = _ws_http
            mlflow.set_experiment("smartshop-llm-finetuning")
            mlflow.start_run(run_name=f"lora-fsdp-{world_size}gpu-actckpt",
                             description=f"LoRA r={lora_r} + FSDP full_shard + activation_checkpointing | "
                                         f"{base_model} | {world_size}x GPU | "
                                         f"max_steps={max_steps} | cosine LR")
            gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"
            gpu_mem_gb = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1) if torch.cuda.is_available() else 0
            mlflow.log_params({
                "base_model": base_model,
                "method": "LoRA+FSDP",
                "lora_r": lora_r,
                "lora_alpha": lora_alpha,
                "lora_target_modules": "q,k,v,o,gate,up,down",
                "parallelism": "FSDP full_shard",
                "precision": "bf16",
                "world_size": world_size,
                "num_nodes": int(os.environ.get("PET_NNODES", 1)) if os.environ.get("PET_NNODES", "1").isdigit() else world_size // torch.cuda.device_count(),
                "gpus_per_node": torch.cuda.device_count(),
                "gpu_type": gpu_name, "gpu_mem_gb": gpu_mem_gb,
                "epochs": epochs,
                "max_steps": max_steps,
                "per_device_batch_size": batch_size,
                "gradient_accumulation": gradient_accumulation,
                "effective_batch_size": batch_size * gradient_accumulation * world_size,
                "learning_rate": lr,
                "weight_decay": 0.01,
                "lr_scheduler": "cosine",
                "warmup_ratio": 0.1,
                "max_seq_length": max_seq_length,
                "total_params": total_params,
                "trainable_params": trainable_params,
                "trainable_pct": f"{trainable_pct:.2f}%",
                "train_examples": len(train_dataset),
                "val_examples": len(val_dataset) if val_dataset else 0,
                "data_source": data_dir,
                "max_files": max_files,
                "bf16": True,
                "activation_checkpointing": True,
                "jit_checkpoint": True,
            })
            mlflow.set_tags({
                "framework": "transformers+trl+peft",
                "distributed": "FSDP",
                "platform": "Red Hat OpenShift AI",
                "task": "causal-lm-finetuning",
            })
            use_mlflow = True
        except Exception as e:
            print(f"MLflow init failed (non-fatal): {e}")

    if rank == 0 and use_mlflow:
        try:
            with mlflow.start_span(name="data_loading") as _sp:
                _sp.set_inputs({"data_dir": data_dir, "max_files": max_files, "max_seq_length": max_seq_length})
                _sp.set_outputs({"train_examples": len(train_dataset),
                                 "val_examples": len(val_dataset) if val_dataset else 0})
            with mlflow.start_span(name="model_init") as _sp:
                _sp.set_inputs({"base_model": base_model, "lora_r": lora_r, "lora_alpha": lora_alpha})
                _sp.set_outputs({"total_params": total_params, "trainable_params": trainable_params,
                                 "trainable_pct": f"{trainable_pct:.2f}%",
                                 "parallelism": "FSDP full_shard", "activation_checkpointing": True,
                                 "gpu_type": gpu_name, "gpu_mem_gb": gpu_mem_gb})
        except Exception:
            pass

    # --- SFT training with FSDP full_shard ---
    training_args = SFTConfig(
        output_dir="/tmp/llm-checkpoints",
        num_train_epochs=epochs,
        max_steps=max_steps,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        gradient_accumulation_steps=gradient_accumulation,
        learning_rate=lr,
        weight_decay=0.01,
        warmup_ratio=0.1,
        lr_scheduler_type="cosine",
        logging_steps=10,
        eval_strategy="no",
        eval_steps=None,
        save_strategy="steps",
        save_steps=300,
        save_total_limit=2,
        bf16=True,
        gradient_checkpointing=False,
        report_to="none",
        dataloader_num_workers=2,
        max_length=max_seq_length,
        fsdp="full_shard auto_wrap",
        fsdp_config={
            "transformer_layer_cls_to_wrap": ["MistralDecoderLayer"],
            "activation_checkpointing": True,
        },
    )

    # Live MLflow callback — streams metrics every logging_steps instead of post-hoc
    import math
    from transformers import TrainerCallback
    class _MLflowStreamCallback(TrainerCallback):
        def __init__(self, active):
            self._active = active
        def on_log(self, args, state, control, logs=None, **kwargs):
            if not self._active or rank != 0:
                return
            step = state.global_step
            m = {}
            if "loss" in (logs or {}):
                m["step_loss"] = logs["loss"]
            if "eval_loss" in (logs or {}):
                m["eval_loss"] = logs["eval_loss"]
                m["eval_perplexity"] = math.exp(min(logs["eval_loss"], 20))
            if "learning_rate" in (logs or {}):
                m["learning_rate"] = logs["learning_rate"]
            if "grad_norm" in (logs or {}):
                m["grad_norm"] = logs["grad_norm"]
            if m:
                try:
                    mlflow.log_metrics(m, step=step)
                except Exception:
                    pass
        def on_evaluate(self, args, state, control, metrics=None, **kwargs):
            if not self._active or rank != 0:
                return
            step = state.global_step
            m = {}
            for k, v in (metrics or {}).items():
                if k.startswith("eval_"):
                    m[k] = v
            if "eval_loss" in m:
                m["eval_perplexity"] = math.exp(min(m["eval_loss"], 20))
            if m:
                try:
                    mlflow.log_metrics(m, step=step)
                except Exception:
                    pass

    hf_trainer = SFTTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        processing_class=tokenizer,
        formatting_func=format_instruction,
        callbacks=[_MLflowStreamCallback(use_mlflow)],
    )

    train_start = _time.time()
    train_result = hf_trainer.train()
    total_time = _time.time() - train_start

    # --- Log training results to MLflow ---
    if rank == 0:
        metrics = train_result.metrics
        train_loss = metrics.get("train_loss", 0)
        train_steps = metrics.get("train_steps", max_steps)
        tokens_per_sec = (
            train_steps * batch_size * gradient_accumulation * max_seq_length * world_size / total_time
            if total_time > 0 else 0
        )
        peak_gpu_mem_gb = round(torch.cuda.max_memory_allocated() / 1e9, 2) if torch.cuda.is_available() else 0

        print(f"Training complete in {total_time:.0f}s")
        print(f"  train_loss={train_loss:.4f}, steps={train_steps}")
        print(f"  throughput ~{tokens_per_sec:,.0f} tokens/s, peak GPU mem: {peak_gpu_mem_gb} GB")

        if use_mlflow:
            mlflow.log_metrics({
                "train_loss": train_loss,
                "total_training_time_s": total_time,
                "throughput_tokens_per_sec": tokens_per_sec,
                "train_steps_completed": train_steps,
                "peak_gpu_memory_gb": peak_gpu_mem_gb,
            })
            try:
                with mlflow.start_span(name="sft_training") as _sp:
                    _sp.set_inputs({"max_steps": max_steps, "batch_size": batch_size,
                                    "gradient_accumulation": gradient_accumulation,
                                    "effective_batch_size": batch_size * gradient_accumulation * world_size})
                    _sp.set_outputs({"train_loss": train_loss, "steps_completed": train_steps,
                                     "total_time_s": round(total_time, 1),
                                     "tokens_per_sec": round(tokens_per_sec),
                                     "peak_gpu_mem_gb": peak_gpu_mem_gb})
            except Exception:
                pass
            if val_dataset:
                quick_eval_ds = val_dataset.select(range(min(50, len(val_dataset))))
                hf_trainer.eval_dataset = quick_eval_ds
                eval_result = hf_trainer.evaluate()
                eval_loss = eval_result.get("eval_loss", 0)
                perplexity = math.exp(min(eval_loss, 20))
                mlflow.log_metrics({
                    "final_eval_loss": eval_loss,
                    "final_perplexity": perplexity,
                })
                print(f"  eval_loss={eval_loss:.4f}, perplexity={perplexity:.2f} (50-sample quick eval)")

    # --- Save adapter to S3 ---
    if rank == 0:
        is_s3 = output_dir.startswith("s3://")
        local_dir = tempfile.mkdtemp(prefix="llm-adapter-") if is_s3 else output_dir
        hf_trainer.save_model(local_dir)
        tokenizer.save_pretrained(local_dir)
        if is_s3:
            fs, _ = fsspec.core.url_to_fs(output_dir, **({"endpoint_url": s3_endpoint} if s3_endpoint else {}))
            fs.put(local_dir, output_dir, recursive=True)
            print(f"LoRA adapter uploaded to {output_dir}")
        else:
            print(f"LoRA adapter saved to {local_dir}")

        if use_mlflow:
            try:
                import json as _json, requests as _req
                _req.packages.urllib3.disable_warnings()
                _run_id = mlflow.active_run().info.run_id
                _exp_id = mlflow.active_run().info.experiment_id
                _ws = workspace or "smartshop"
                _base = tracking_uri
                _tok = os.environ.get("MLFLOW_TRACKING_TOKEN", "")
                _headers = {
                    "Authorization": f"Bearer {_tok}",
                    "X-MLflow-Workspace": _ws,
                    "Content-Type": "application/octet-stream",
                }
                adapter_cfg_path = os.path.join(local_dir, "adapter_config.json")
                if os.path.exists(adapter_cfg_path):
                    with open(adapter_cfg_path, "rb") as _af:
                        _art_url = f"{_base}/api/2.0/mlflow-artifacts/artifacts/workspaces/{_ws}/{_exp_id}/{_run_id}/artifacts/adapter/adapter_config.json"
                        _r = _req.put(_art_url, data=_af.read(), headers=_headers, verify=False, timeout=30)
                        print(f"  adapter_config.json upload: {_r.status_code}")
                _card = {
                    "model_name": "smartshop-llm-mistral-fsdp-lora",
                    "base_model": base_model,
                    "framework": "transformers+peft",
                    "adapter_uri": output_dir,
                    "training": {"method": "LoRA+FSDP", "lora_r": lora_r, "lora_alpha": lora_alpha,
                                 "max_steps": max_steps, "batch_size": batch_size},
                }
                _card_url = f"{_base}/api/2.0/mlflow-artifacts/artifacts/workspaces/{_ws}/{_exp_id}/{_run_id}/artifacts/adapter/model_card.json"
                _req.put(_card_url, data=_json.dumps(_card, indent=2).encode(),
                         headers=_headers, verify=False, timeout=30)
                mlflow.set_tag("adapter_s3_uri", output_dir)
                mlflow.set_tag("base_model", base_model)
                try:
                    with mlflow.start_span(name="model_save_and_register") as _sp:
                        _sp.set_inputs({"output_dir": output_dir})
                        _sp.set_outputs({"adapter_location": output_dir,
                                         "peak_gpu_mem_gb": peak_gpu_mem_gb,
                                         "total_time_s": round(total_time, 1)})
                except Exception:
                    pass
                try:
                    mlflow.pyfunc.log_model(
                        artifact_path="model",
                        python_model=mlflow.pyfunc.PythonModel(),
                        registered_model_name="smartshop-llm-mistral-fsdp-lora",
                    )
                    print("  Logged model to MLflow Models tab")
                except Exception as _lme:
                    print(f"  log_model partial (non-fatal): {_lme}")
                from mlflow.tracking import MlflowClient
                _client = MlflowClient()
                try:
                    _client.create_registered_model("smartshop-llm-mistral-fsdp-lora",
                        tags={"task": "llm-finetuning", "framework": "transformers+peft+fsdp"},
                        description="Mistral-7B LoRA+FSDP adapter for product Q&A")
                except Exception:
                    pass
                _client.create_model_version(
                    name="smartshop-llm-mistral-fsdp-lora",
                    source=output_dir,
                    run_id=_run_id,
                    description=f"LoRA+FSDP r={lora_r} alpha={lora_alpha}")
                print("  Registered model version: smartshop-llm-mistral-fsdp-lora")
            except Exception as _me:
                print(f"  Model registration failed (non-fatal): {_me}")
            mlflow.set_tag("training_status", "success")
            mlflow.set_tag("run_summary",
                           f"loss={train_loss:.4f} | "
                           f"{round(total_time/60,1)}min | "
                           f"{round(tokens_per_sec):,} tok/s | "
                           f"peak_gpu={peak_gpu_mem_gb}GB")
            mlflow.end_run()
            print("Model registered in MLflow as: smartshop-llm-mistral-fsdp-lora")

print("train_llm defined")

## Submit TrainJob

In [ ]:
import base64
from datetime import datetime
from kubeflow.trainer.rhai import TransformersTrainer
from kubeflow.trainer.rhai.transformers import PeriodicCheckpointConfig
from kubeflow.trainer.options import (
    Name, Labels, PodTemplateOverrides, PodTemplateOverride,
    PodSpecOverride, ContainerOverride
)

# Read secrets
v1 = k8s.CoreV1Api(k8s.ApiClient(cfg) if cfg else k8s.ApiClient())
s3_secret = v1.read_namespaced_secret(S3_CREDENTIALS_SECRET, NAMESPACE)
mlflow_secret = v1.read_namespaced_secret(MLFLOW_SECRET, NAMESPACE)
hf_secret = v1.read_namespaced_secret(HF_SECRET, NAMESPACE)

def decode_secret(secret, key):
    return base64.b64decode(secret.data[key]).decode()

job_id = datetime.now().strftime('%m%d-%H%M')
JOB_NAME = f"llm-finetune-{job_id}"

# Guard: skip if an llm-finetune job is already running or completed
_custom_api = k8s.CustomObjectsApi(k8s.ApiClient(cfg) if cfg else k8s.ApiClient())
try:
    _jobs = _custom_api.list_namespaced_custom_object(
        "trainer.kubeflow.org", "v1", NAMESPACE, "trainjobs"
    ).get("items", [])
    _active = [
        j["metadata"]["name"] for j in _jobs
        if j["metadata"]["name"].startswith("llm-finetune-")
        and not any(
            c.get("type") == "Failed" and c.get("status") == "True"
            for c in (j.get("status", {}).get("conditions") or [])
        )
    ]
except Exception:
    _active = []

if _active:
    JOB_NAME = _active[0]
    print(f"Active TrainJob already exists: {JOB_NAME} — skipping submission")
    _SKIP_SUBMIT = True
else:
    _SKIP_SUBMIT = False

func_args = {
    'data_dir': DATA_DIR,
    'output_dir': OUTPUT_DIR,
    'base_model': BASE_MODEL,
    'epochs': EPOCHS,
    'max_steps': MAX_STEPS,
    'batch_size': BATCH_SIZE,
    'gradient_accumulation': GRADIENT_ACCUMULATION,
    'lr': LR,
    'lora_r': LORA_R,
    'lora_alpha': LORA_ALPHA,
    'max_seq_length': MAX_SEQ_LENGTH,
    'max_files': MAX_FILES,
}

if not _SKIP_SUBMIT:
    job = trainer.train(
        trainer=TransformersTrainer(
            func=train_llm,
            func_args=func_args,
            num_nodes=NUM_NODES,
            resources_per_node={'gpu': GPUS_PER_NODE, 'cpu': 8, 'memory': '128Gi'},
            env={
                'MLFLOW_TRACKING_URI': MLFLOW_TRACKING_URI,
                'MLFLOW_TRACKING_INSECURE_TLS': 'true',
                'MLFLOW_TRACKING_TOKEN': decode_secret(mlflow_secret, 'MLFLOW_TRACKING_TOKEN'),
                'MLFLOW_WORKSPACE': decode_secret(mlflow_secret, 'MLFLOW_WORKSPACE'),
                'AWS_ENDPOINT_URL_S3': MINIO_ENDPOINT,
                'AWS_S3_ENDPOINT': MINIO_ENDPOINT,
                'S3_ENDPOINT': MINIO_ENDPOINT,
                'HF_TOKEN': decode_secret(hf_secret, 'token'),
                'HF_HOME': '/tmp/hf_home',
                'NCCL_DEBUG': 'INFO',
                'NCCL_IB_DISABLE': '1',
            },
            enable_progression_tracking=True,
            output_dir="s3://smartshop-models/llm-checkpoints",
            data_connection_name="smartshop-s3-credentials",
            periodic_checkpoint_config=PeriodicCheckpointConfig(
                save_strategy="steps",
                save_steps=300,
                save_total_limit=2,
            ),
            verify_cloud_storage_ssl=False,
        ),
        runtime=runtime,
        options=[
            Name(JOB_NAME),
            Labels({'app': 'smartshop', 'component': 'llm-training'}),
            PodTemplateOverrides(PodTemplateOverride(
                target_jobs=['node'],
                spec=PodSpecOverride(
                    volumes=[
                        {'name': 'shm', 'emptyDir': {'medium': 'Memory'}},
                    ],
                    containers=[ContainerOverride(
                        name='node',
                        volume_mounts=[
                            {'name': 'shm', 'mountPath': '/dev/shm'},
                        ],
                    )]
                )
            ))
        ]
    )
    print(f"Submitted: {JOB_NAME}")
else:
    print(f"Reusing existing job: {JOB_NAME}")

## Monitor Training

In [ ]:
print(f"Waiting for {JOB_NAME} to start running...")
trainer.wait_for_job_status(name=JOB_NAME, status={"Running"}, timeout=600)
print(f"{JOB_NAME} is Running")

print(f"Waiting for completion (timeout {TIMEOUT_SECONDS}s)...")
trainer.wait_for_job_status(name=JOB_NAME, status={"Complete", "Failed"}, timeout=TIMEOUT_SECONDS)

status = trainer.get_job(JOB_NAME)
print(f"Final status: {status.status}")

_failed = False
if hasattr(status, 'status') and hasattr(status.status, 'conditions'):
    conditions = {c.type: c.status for c in (status.status.conditions or [])}
    if conditions.get('Failed') == 'True':
        _failed = True
        print("TrainJob reported Failed — checking if adapter was actually saved...")
        try:
            os.environ['AWS_ENDPOINT_URL_S3'] = MINIO_ENDPOINT
            import fsspec as _fs
            _s3, _ = _fs.core.url_to_fs(OUTPUT_DIR, endpoint_url=MINIO_ENDPOINT)
            _files = _s3.ls(OUTPUT_DIR)
            if any('adapter_config.json' in f for f in _files):
                print("Adapter found on S3 — training succeeded despite exit barrier error.")
                _failed = False
            else:
                print("No adapter found — genuine failure.")
        except Exception as _e:
            print(f"S3 check failed: {_e}")

if _failed:
    print("Last 30 lines of logs:")
    for line in list(trainer.get_job_logs(JOB_NAME, follow=False))[-30:]:
        print(line)
    raise RuntimeError(f"TrainJob {JOB_NAME} failed")

print(f"TrainJob {JOB_NAME} completed successfully!")

In [ ]:
print('NOTEBOOK_STATUS: SUCCESS')